# Regime-switching linear combination

As the linear combination, but the weights, constant and sigma depend on a regime that follows a Markov chain, shared by all columns and moving once per row. The regime path is returned as hidden state `regime`.

Edit the parameters cell and re-run all: the same parameters and seeds always give the same paths.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import fantasma as fm

## Parameters

Weights, constant and sigma are each a scalar (all columns), a list with one value per column, or a Series keyed by column. `sigma` is annualised; the noise each row is `sigma * sqrt(dt) * eps`, with `dt` the time since the previous row in years.

In [ ]:
# Input series: N_INPUTS frames with the same index and columns. Here they are simulated
# random walks in log price, one column per asset; any list of such frames works, e.g. real
# data from Almanac: [load(...)["close"].unstack("symbol"), ...].
INPUT_SEED = 1
N_INPUTS = 2
ASSETS = ["X", "Y"]
INPUT_SIGMA = 0.2
START = "2024-01-01"
FREQ = "1d"
N_STEPS = 365 * 3

# Simulation
SEED = 42
N_PATHS = 3

# Model: one linear combination per regime
REGIMES = [
    dict(weights=[1.0, 0.0], constant=0.0, sigma=0.05),  # regime 0: follows input 0
    dict(weights=[0.0, 1.0], constant=0.5, sigma=0.20),  # regime 1: follows input 1, noisier
]
# Markov chain: TRANSITION_MATRIX[i][j] = probability of moving from regime i to j in one row
TRANSITION_MATRIX = [
    [0.98, 0.02],
    [0.05, 0.95],
]
INITIAL_REGIME = None  # an int, a probability per regime, or None for the stationary distribution


## Inputs

In [ ]:
input_model = fm.Composite({a: fm.RandomWalk(INPUT_SIGMA, asset=a) for a in ASSETS})
inputs = [
    np.log(p)
    for p in fm.simulate(
        input_model, start=START, freq=FREQ, n_steps=N_STEPS, seed=INPUT_SEED, n_paths=N_INPUTS
    )
]
inputs[0].head()

## Simulate

In [ ]:
model = fm.RegimeSwitching(
    [fm.LinearCombination(**r) for r in REGIMES],
    transition_matrix=TRANSITION_MATRIX,
    initial=INITIAL_REGIME,
)
print("stationary distribution:", np.round(model.chain.stationary_distribution(), 4))
model

In [ ]:
paths = fm.simulate_driven(model, inputs, seed=SEED, n_paths=N_PATHS, return_hidden=True)
out, hidden = paths[0]
pd.concat([out, hidden], axis=1).head()

In [ ]:
fig, axes = plt.subplots(len(ASSETS) + 1, 1, figsize=(10, 2.5 * (len(ASSETS) + 1)), sharex=True)
regime = hidden["regime"]
for ax, asset in zip(axes[:-1], ASSETS):
    for i, x in enumerate(inputs):
        ax.plot(x.index, x[asset], lw=0.8, alpha=0.6, label=f"input {i}")
    ax.plot(out.index, out[asset], lw=0.8, color="k", label="output (path 0)")
    ax.set_title(asset)
    ax.legend(fontsize=8)
axes[-1].step(regime.index, regime, where="post", lw=0.8)
axes[-1].set_yticks(range(len(REGIMES)))
axes[-1].set_title("regime (path 0)")
plt.show()

## Check: transition frequencies

The observed transitions between regimes estimate the transition matrix.

In [ ]:
n = len(REGIMES)
counts = np.zeros((n, n))
for _, h in paths:
    r = h["regime"].to_numpy()
    np.add.at(counts, (r[:-1], r[1:]), 1)
print("true:\n", np.asarray(TRANSITION_MATRIX))
print("estimated from all paths:\n", np.round(counts / counts.sum(axis=1, keepdims=True), 3))